# Leren 경제·금융 데이터셋 EDA 및 정밀도 실험

이 노트북은 KDI·BOK 라벨링 Excel 320건 통합 평가 EDA, 기존 정밀도 파이프라인, KDI·한국은행 전체 데이터셋 EDA를 구분해 실행합니다.

- 문제 생성: 독립 Gold가 승인된 경우 관련도 2를 양성으로 사용하여 Precision을 우선 확인합니다.
- 튜터 검색: 독립 Gold가 승인된 경우 관련도 1 또는 2를 양성으로 사용하여 Recall을 함께 확인합니다.
- 키워드 매칭은 최종 판정이 아니라 후보 추출 단계입니다.
- 기본 통합 평가 셀은 외부 API를 호출하지 않고 기존 LLM 파일럿 결과만 진단적으로 재사용합니다.
- 독립 검증 전의 현재 골드는 참고 기준으로만 사용하며, LLM 결과만으로 Precision·Recall을 확정하지 않습니다.

In [1]:
# ruff: noqa
import os
import sys
from pathlib import Path

# 노트북을 저장소 루트·notebook 디렉터리 중 어디에서 실행해도 동작하게 합니다.
ROOT = Path.cwd()
while not (ROOT / 'app').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
# Settings가 저장소 루트의 .env를 읽도록 Notebook 작업 디렉터리를 고정합니다.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json

from scripts.run_kdi_bok_evaluation import run as run_integrated_evaluation

# 기존 단일 workbook 기본 탐색은 사용하지 않고, KDI/BOK workbook을 함께 읽습니다.
# API 호출은 명시적으로 True로 바꿀 때만 최대 20건 실행합니다.
RUN_LLM_API = False
LLM_SAMPLE_SIZE = 20
integrated_run = run_integrated_evaluation(
    output_dir=ROOT / 'reports' / 'evaluation',
    combined_path=ROOT / 'data' / 'processed' / 'evaluation' / 'combined_320.jsonl',
    pilot_path=ROOT / 'data' / 'labeled' / 'precision' / 'llm_pilot_results.jsonl',
    data_dir=ROOT / 'data',
    execute_llm=RUN_LLM_API,
    llm_sample_size=LLM_SAMPLE_SIZE,
)
eda_summary = json.loads((ROOT / 'reports' / 'evaluation' / 'precision_eda_summary.json').read_text())
print('KDI 라벨링 행 수:', eda_summary['dataset']['labeled_sample_count_by_source']['KDI'])
print('BOK 라벨링 행 수:', eda_summary['dataset']['labeled_sample_count_by_source']['BOK'])
print('통합 라벨링 행 수:', eda_summary['dataset']['labeled_sample_count'])
print('전체 canonical 문서 수:', eda_summary['dataset']['canonical_document_count'])

KDI 라벨링 행 수: 160
BOK 라벨링 행 수: 160
통합 라벨링 행 수: 320
전체 canonical 문서 수: 3539


## 0. LLM 소규모 파일럿 실험

현재 EDA는 키워드 기준선과 LLM 파일럿을 함께 확인합니다. API 파일럿은 KDI/BOK 후보·비후보를 층화하여 최대 20건만 실행합니다.

API 호출은 첫 번째 실행 셀의 `RUN_LLM_API=True`일 때만 발생합니다. 결과는 Git에 올리지 않는 로컬 JSONL 파일에 저장되며, 이 결과만으로 최종 Precision·Recall을 주장하지 않습니다.

In [2]:
# API 실행은 첫 번째 셀의 RUN_LLM_API 토글로 한 곳에서만 제어합니다.
method_report = json.loads((ROOT / 'reports' / 'evaluation' / 'method_comparison.json').read_text())
execution = method_report.get('llm_execution', {})
print('요청 모델:', method_report.get('model') or '설정값 확인 필요')
print('파일럿 상태:', execution.get('status', method_report.get('status')))
print('파일럿 표본 수:', execution.get('sample_size', 0))
print('이번 실행 API 호출 수:', execution.get('llm_calls', 0))
print('기록된 API 호출 수:', execution.get('recorded_api_calls', execution.get('llm_calls', 0)))


요청 모델: ['openai/gpt-5.6-sol']
파일럿 상태: REUSED_EXECUTED_LLM_PILOT
파일럿 표본 수: 20
이번 실행 API 호출 수: 0
기록된 API 호출 수: 20


## 1. 데이터 규모·토큰·라벨 분포


In [3]:
# ruff: noqa: E501
dataset = eda_summary['dataset']
labels = eda_summary['labels']
print(f"전체 corpus: {dataset['canonical_document_count']}건 / KDI {dataset['canonical_document_count_by_source']['KDI']}건 / BOK {dataset['canonical_document_count_by_source']['BOK']}건")
print(f"라벨링 표본: {dataset['labeled_sample_count']}건 / KDI {dataset['labeled_sample_count_by_source']['KDI']}건 / BOK {dataset['labeled_sample_count_by_source']['BOK']}건")
print(f"청크: {dataset['canonical_chunk_count']}개, 토큰: {dataset['canonical_token_total_from_latest_eda']:,}개")
print(f"A/B 일치율: KDI {labels['agreement']['KDI']['agreement_rate']:.1%}, BOK {labels['agreement']['BOK']['agreement_rate']:.1%}, 전체 {labels['agreement']['ALL']['agreement_rate']:.1%}")
print(f"A/B 불일치: KDI {labels['agreement']['KDI']['disagree_count']}건, BOK {labels['agreement']['BOK']['disagree_count']}건")
print(f"B 재검토 대상: KDI {labels['b_review_flag_count_by_source']['KDI']}건, BOK {labels['b_review_flag_count_by_source']['BOK']}건")


전체 corpus: 3539건 / KDI 2750건 / BOK 789건
라벨링 표본: 320건 / KDI 160건 / BOK 160건
청크: 3545개, 토큰: 770,607개
A/B 일치율: KDI 79.6%, BOK 42.5%, 전체 51.2%
A/B 불일치: KDI 10건, BOK 92건
B 재검토 대상: KDI 50건, BOK 21건


## 2. 기본 키워드와 확장 키워드 비교

기본 키워드는 핵심어만 사용하고, 확장 키워드는 약어·연관어·반대 개념까지 포함합니다. 두 전략을 문제 생성과 튜터 검색 목적별로 따로 평가합니다.

In [4]:
for method_name, method in eda_summary['methods'].items():
    if method_name not in {'keyword_only', 'llm_only', 'hybrid'}:
        continue
    print(f"{method_name:12} | 상태={method['status']} | 평가 문서={method['evaluated_docs']}건")
    for policy_name, policy in [('문제 생성', method['proxy_question']), ('튜터 검색', method['proxy_tutor'])]:
        metric = policy.get('metrics')
        print(f"  {policy_name}: {metric if metric is not None else 'NOT_EVALUATED'} / 모집단 {policy['population_count']}건")
print('주의: 현재 P/R/F1은 독립 Gold가 없는 DRY_RUN proxy이며 최종 성능값이 아닙니다.')

keyword_only | 상태=EXECUTED_OFFLINE | 평가 문서=320건
  문제 생성: {'precision': 0.64, 'recall': 0.6153846153846154, 'f1': 0.6274509803921569} / 모집단 228건
  튜터 검색: {'precision': 0.96, 'recall': 0.22641509433962265, 'f1': 0.366412213740458} / 모집단 228건
llm_only     | 상태=EXECUTED_LLM_PILOT | 평가 문서=20건
  문제 생성: {'precision': 1.0, 'recall': 0.4, 'f1': 0.5714285714285715} / 모집단 11건
  튜터 검색: {'precision': 1.0, 'recall': 0.2857142857142857, 'f1': 0.4444444444444445} / 모집단 11건
hybrid       | 상태=EXECUTED_LLM_PILOT | 평가 문서=279건
  문제 생성: {'precision': 1.0, 'recall': 0.26666666666666666, 'f1': 0.4210526315789474} / 모집단 208건
  튜터 검색: {'precision': 1.0, 'recall': 0.04597701149425287, 'f1': 0.08791208791208792} / 모집단 208건
주의: 현재 P/R/F1은 독립 Gold가 없는 DRY_RUN proxy이며 최종 성능값이 아닙니다.


In [5]:
# ruff: noqa: E501
from IPython.display import Markdown, display

# 그래프 대신 표를 사용해 멘토님과 교수님이 실제 수치를 바로 확인할 수 있도록 합니다.
# 모든 값은 앞 단계에서 Excel과 키워드 평가 결과를 다시 계산한 summary에서 가져옵니다.
dataset = eda_summary['dataset']
labels = eda_summary['labels']

def show_table(headers, rows):
    """Jupyter 기본 Markdown으로 결과 표를 출력합니다."""
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    lines.extend('| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows)
    display(Markdown('\n'.join(lines)))

overview_table = [
    ['전체 corpus', dataset['canonical_document_count'], f"KDI {dataset['canonical_document_count_by_source']['KDI']} / BOK {dataset['canonical_document_count_by_source']['BOK']}"],
    ['라벨링 표본', dataset['labeled_sample_count'], f"KDI {dataset['labeled_sample_count_by_source']['KDI']} / BOK {dataset['labeled_sample_count_by_source']['BOK']}"],
    ['청크 / 토큰', dataset['canonical_chunk_count'], f"{dataset['canonical_token_total_from_latest_eda']:,} tokens"],
    ['A/B 일치율', f"KDI {labels['agreement']['KDI']['agreement_rate']:.1%}", f"BOK {labels['agreement']['BOK']['agreement_rate']:.1%} / 전체 {labels['agreement']['ALL']['agreement_rate']:.1%}"],
    ['A/B 불일치', labels['agreement']['KDI']['disagree_count'], f"BOK {labels['agreement']['BOK']['disagree_count']}건"],
    ['독립 Gold', eda_summary['independent_gold']['count'], eda_summary['independent_gold']['status']],
]
print('### 핵심 데이터 및 검수 지표')
show_table(['지표', '값', '해석'], overview_table)

strategy_rows = []
for method_name in ('keyword_only', 'llm_only', 'hybrid'):
    method = eda_summary['methods'][method_name]
    strategy_rows.append([method_name, method['status'], method['evaluated_docs'], method.get('candidate_docs', method.get('observed_prediction_docs', '-'))])

print('### 키워드 전략 비교')
show_table(['방법', '상태', '평가 문서', '후보/관측 문서'], strategy_rows)

llm_method = eda_summary['methods']['llm_only']
llm_execution = method_report.get('llm_execution', {})
provider_models = llm_method.get('model') or ['확인 불가']
pilot_table = [
    ['요청 모델', ', '.join(provider_models)],
    ['Provider 응답 모델', ', '.join(provider_models)],
    ['파일럿 상태', llm_execution.get('status', 'NOT_EXECUTED')],
    ['이번 실행 호출 수', llm_execution.get('llm_calls', 0)],
    ['기록된 호출 수', llm_execution.get('recorded_api_calls', 0)],
    ['결과 파일', llm_execution.get('output_path', '확인 불가')],
]
print('### LLM 파일럿 실행 현황')
show_table(['항목', '값'], pilot_table)

print('### 의사결정 요약')
print('1. 문제 생성: 확장 키워드로 후보를 넓게 수집한 뒤 LLM 분류와 Human Review를 적용합니다.')
print('2. 튜터 검색: 키워드 필터만으로 제한하지 않고 임베딩 검색과 챕터 태그를 함께 사용합니다.')
print('3. 해석 주의: 현재 Precision·Recall은 독립적으로 확정된 골드가 아닌 Excel 라벨 기준선입니다.')

### 핵심 데이터 및 검수 지표


| 지표 | 값 | 해석 |
| --- | --- | --- |
| 전체 corpus | 3539 | KDI 2750 / BOK 789 |
| 라벨링 표본 | 320 | KDI 160 / BOK 160 |
| 청크 / 토큰 | 3545 | 770,607 tokens |
| A/B 일치율 | KDI 79.6% | BOK 42.5% / 전체 51.2% |
| A/B 불일치 | 10 | BOK 92건 |
| 독립 Gold | 0 | NOT_READY |

### 키워드 전략 비교


| 방법 | 상태 | 평가 문서 | 후보/관측 문서 |
| --- | --- | --- | --- |
| keyword_only | EXECUTED_OFFLINE | 320 | 51 |
| llm_only | EXECUTED_LLM_PILOT | 20 | - |
| hybrid | EXECUTED_LLM_PILOT | 279 | 279 |

### LLM 파일럿 실행 현황


| 항목 | 값 |
| --- | --- |
| 요청 모델 | openai/gpt-5.6-sol |
| Provider 응답 모델 | openai/gpt-5.6-sol |
| 파일럿 상태 | REUSED_EXECUTED_LLM_PILOT |
| 이번 실행 호출 수 | 0 |
| 기록된 호출 수 | 20 |
| 결과 파일 | data/labeled/precision/llm_pilot_results.jsonl |

### 의사결정 요약
1. 문제 생성: 확장 키워드로 후보를 넓게 수집한 뒤 LLM 분류와 Human Review를 적용합니다.
2. 튜터 검색: 키워드 필터만으로 제한하지 않고 임베딩 검색과 챕터 태그를 함께 사용합니다.
3. 해석 주의: 현재 Precision·Recall은 독립적으로 확정된 골드가 아닌 Excel 라벨 기준선입니다.


## 3. 실험 해석 및 다음 단계

현재 결과에서는 확장 키워드가 문제 생성 후보의 재현율을 높였지만, 튜터 검색 목적에는 여전히 재현율이 낮습니다. 따라서 문제 생성은 키워드 후보 뒤에 LLM 정밀 분류와 Human Review를 두고, 튜터 검색은 키워드로 과도하게 제한하지 않고 임베딩 검색과 챕터 태그를 함께 사용하는 방향이 적절합니다.

현재 Workbook 라벨과 A/B 합의는 독립 인간 Gold가 아니므로 현재 수치는 모델의 독립 성능이 아닙니다. KDI A/B 불일치 10건, BOK A/B 불일치 92건, B 재검토 대상 21건, 자료 품질 검토필요 행 및 직접 관련 양성 표본을 포함한 독립 Gold 세트를 확정한 뒤 같은 Notebook에서 임계값별 Precision·Recall 변화를 비교해야 합니다.

## 4. Keyword-only·LLM-only·Hybrid 비교 실험 준비

이 셀은 KDI 160건과 BOK 160건을 합친 동일한 독립 Gold Test Set에서 세 방법을 비교하기 위한 실행 준비 상태를 확인합니다. 현재 두 Workbook 라벨은 독립 Gold Set이 아니므로, Gold Set이 없으면 Precision·Recall·F1과 비교 그래프를 생성하지 않습니다.

- Keyword-only: 확장 키워드 후보 여부를 사용합니다.
- LLM-only: relevance_score 임계값을 전체 평가 대상에 적용합니다.
- Hybrid: 키워드 후보에는 LLM 임계값을 적용하고, 후보 밖 문서는 명시적 음성으로 기록하여 전체 Gold 모집단을 평가합니다.
- 모든 LLM 결과는 사람 검수 전 PENDING 상태로 유지합니다.

In [6]:
# ruff: noqa: E501
# 위 통합 실행 셀에서 생성한 동일한 320건 평가 결과를 사용합니다.
# KDI 단일 workbook을 다시 선택하는 기존 compare_precision_methods 호출은 제거했습니다.
method_comparison = json.loads((ROOT / 'reports' / 'evaluation' / 'method_comparison.json').read_text())
gold_status = method_comparison['independent_gold']['status']
print('세 방법 비교 상태:', method_comparison['status'])
print('독립 Gold Set 상태:', gold_status)
for method_name in ('keyword_only', 'llm_only', 'hybrid'):
    method = method_comparison['methods'][method_name]
    print(f"{method_name}: 평가 문서 {method['coverage_count']}건 / 전체 모집단 {method['population_count']}건 / 상태 {method['status']}")
print('신규 API 호출:', method_comparison['new_api_calls'])
print('기록된 API 호출:', method_comparison.get('llm_execution', {}).get('recorded_api_calls', 0))
print('비용 상태:', method_comparison['cost']['status'])
print('독립 Gold가 없어 최종 성능 지표와 그래프는 생성하지 않았습니다.')


세 방법 비교 상태: DRY_RUN_AI_REFERENCE
독립 Gold Set 상태: NOT_READY
keyword_only: 평가 문서 320건 / 전체 모집단 320건 / 상태 EXECUTED_OFFLINE
llm_only: 평가 문서 20건 / 전체 모집단 320건 / 상태 EXECUTED_LLM_PILOT
hybrid: 평가 문서 279건 / 전체 모집단 320건 / 상태 EXECUTED_LLM_PILOT
신규 API 호출: 0
기록된 API 호출: 20
비용 상태: UNCONFIRMED_CONFIG_REQUIRED
독립 Gold가 없어 최종 성능 지표와 그래프는 생성하지 않았습니다.


## 5. 독립 Gold 검수 큐와 키워드 누락 진단

현재 엑셀의 확정 골드는 기존 라벨을 참고하기 위한 값이며, 독립 Gold로 자동 승격하지 않습니다. 아래 셀은 원문·ID·Content Hash를 확인할 수 있는 검수 큐를 만들고, 개발셋과 테스트셋을 안정적인 해시로 분리합니다. 검수자가 합의한 행만 `INDEPENDENT_APPROVED`와 `APPROVED`로 변경해야 비교 지표가 활성화됩니다.

키워드 미매칭 분석의 기존 라벨 점수는 사전 확장에 활용할 수 있는 참고 진단일 뿐이며, 최종 테스트셋 튜닝이나 Precision·Recall 산출에는 사용하지 않습니다.

In [7]:
# ruff: noqa: E501
from scripts.prepare_independent_gold_review import run as prepare_gold_review

# 원본 엑셀은 읽기만 하고, 검수 큐와 보고서를 별도 경로에 생성합니다.
review_artifacts = prepare_gold_review()
review_status = review_artifacts['status']
diagnostic = review_artifacts['diagnostic']
print('독립 Gold 평가 상태:', review_status['status'])
print('검수 큐 행 수:', review_status['queue_row_count'])
print('개발셋/테스트셋:', review_status['split_counts'])
print('출처 그룹:', review_status['source_groups_supported_by_queue'])
print('BOK 입력 포함 여부:', review_status['bok_present_in_current_input'])
print('기존 참고 라벨 기준 미매칭 관련 후보:', diagnostic['reference_positive_misses_count'])
print('검수 큐:', review_status['review_queue'])
print('누락 분석 보고서:', review_status['diagnostic_report'])

독립 Gold 평가 상태: NOT_EVALUATED
검수 큐 행 수: 320
개발셋/테스트셋: {'development': 270, 'test': 50}
출처 그룹: ['BOK', 'KDI']
BOK 입력 포함 여부: True
기존 참고 라벨 기준 미매칭 관련 후보: 57
검수 큐: data/labeled/precision/independent_gold_review_queue.csv
누락 분석 보고서: reports/keyword_miss_analysis.json


## 6. KDI·한국은행 전체 데이터셋 EDA

라벨링 표본 320건과 별도로, 실제 KDI 시사경제용어와 한국은행 경제금융용어 전체 데이터를 함께 분석합니다. JSON과 JSONL은 같은 기관의 미러 파일이므로 JSONL을 canonical 파일로 사용하고 중복 집계하지 않습니다.

이 셀은 외부 LLM을 호출하지 않습니다. 전체 문서의 토큰 수, 기관별 규모, 청크 수, 품질 이슈, 인플레이션 후보 수를 오프라인으로 계산합니다.

In [8]:
from scripts.keyword_filter import run as run_full_keywords
from scripts.make_chunks import run as make_full_chunks
from scripts.run_eda import run as run_full_eda

# 원본 파일이 저장소 루트의 KDI·한국은행 폴더에 있어 ROOT를 입력으로 사용합니다.
full_eda = run_full_eda(
    ROOT,
    ROOT / 'configs' / 'eda_config.yaml',
    ROOT / 'configs' / 'chapters.yaml',
    ROOT / 'reports' / 'pipeline',
)
full_chunks = make_full_chunks(
    ROOT,
    ROOT / 'configs' / 'eda_config.yaml',
    ROOT / 'data' / 'processed',
)
full_keyword = run_full_keywords(
    ROOT,
    ROOT / 'configs' / 'eda_config.yaml',
    ROOT / 'configs' / 'chapters.yaml',
    ROOT / 'data' / 'labeled' / 'keyword',
    ROOT / 'data' / 'processed' / 'chunks.jsonl',
)
print('전체 고유 문서 수:', full_eda['corpus']['unique_document_count'])
print('기관별 문서 수:', full_eda['corpus']['document_count_by_source'])
print('전체 평균 토큰 수:', round(full_eda['corpus']['token_count']['average'], 2))
print('전체 토큰 수:', full_eda['corpus']['token_count']['total'])
print('청크 수:', full_chunks['chunk_count'])
print('기관별 청크 수:', full_chunks['chunk_count_by_source'])
print('키워드 후보 청크 수:', full_keyword['candidate_chunk_count'])
print('키워드 후보 비율:', f"{full_keyword['candidate_chunk_rate']:.2%}")
print('품질 이슈:', [row for row in full_eda['quality']['rows'] if row['count']])
print('상세 EDA 보고서: reports/pipeline/eda_report.md')

전체 고유 문서 수: 3539
기관별 문서 수: {'BOK': 789, 'KDI': 2750}
전체 평균 토큰 수: 217.75
전체 토큰 수: 770607
청크 수: 3545
기관별 청크 수: {'BOK': 791, 'KDI': 2754}
키워드 후보 청크 수: 148
키워드 후보 비율: 4.17%
품질 이슈: [{'issue_type': 'missing_source_url', 'count': 789, 'status': 'REVIEW'}, {'issue_type': 'unknown_source_review_status', 'count': 3539, 'status': 'REVIEW'}, {'issue_type': 'unconfirmed_license_status', 'count': 3539, 'status': 'REVIEW'}, {'issue_type': 'very_short_under_20_tokens', 'count': 7, 'status': 'REVIEW'}, {'issue_type': 'longer_than_chunk_limit', 'count': 6, 'status': 'REVIEW'}, {'issue_type': 'duplicate_content_hash_within_source', 'count': 1, 'status': 'FAIL'}]
상세 EDA 보고서: reports/pipeline/eda_report.md


## 7. KDI·BOK 320건 통합 분류 실험 EDA

KDI 160건과 BOK 160건의 실제 Excel을 canonical 원본과 대조한 뒤, 기존 Keyword 분류기와 저장된 LLM 파일럿을 연결합니다. BOK의 `확정 골드=완료`는 숫자 라벨로 변환하지 않으며, A/B 합의는 `REVIEWER_AB_CONSENSUS_PROXY`로만 표시합니다. 독립 Gold가 없으면 최종 Precision·Recall·F1은 생성하지 않습니다.

이 셀은 외부 API를 호출하지 않습니다. 재현 실행기와 보고서의 JSON/CSV/PNG 산출물을 다시 읽어 EDA 표를 구성합니다.

In [9]:
# ruff: noqa: E501
import csv
import json
from pathlib import Path

from IPython.display import display

# 첫 번째 실행 셀이 산출물을 생성합니다. 이 섹션은 동일 산출물을 다시 호출하지 않고 읽습니다.
integrated_run = None
evaluation_dir = ROOT / 'reports' / 'evaluation'
combined_summary = json.loads((evaluation_dir / 'combined_320_summary.json').read_text())
agreement_report = json.loads((evaluation_dir / 'label_agreement_report.json').read_text())
method_report = json.loads((evaluation_dir / 'method_comparison.json').read_text())
print('통합 상태:', combined_summary['status'])
print('기관별 통합 건수:', combined_summary['source_row_counts'])
print('전체 corpus:', combined_summary['corpus_inventory']['document_count'])
print('원문 대조:', {source: report['reconciliation_counts'] for source, report in combined_summary['source_reports'].items()})
for source in ('KDI', 'BOK', 'ALL'):
    item = agreement_report['by_source'][source]
    print(f"{source} A/B: {item['agree_count']} 일치, {item['disagree_count']} 불일치, 일치율={item['agreement_rate']:.2%}")
print('독립 Gold:', method_report['independent_gold'])
print('신규 API 호출:', method_report['new_api_calls'])
print('기록된 API 호출:', method_report.get('llm_execution', {}).get('recorded_api_calls', 0))
print('방법별 상태/커버리지:', {name: (item['status'], item['coverage_count']) for name, item in method_report['methods'].items() if name in ('keyword_only', 'llm_only', 'hybrid')})
def preview_csv(path, limit=10):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))[:limit]
display(preview_csv(evaluation_dir / 'label_disagreements.csv'))
display(preview_csv(evaluation_dir / 'keyword_predictions.csv'))
eda_summary = json.loads((evaluation_dir / 'precision_eda_summary.json').read_text())
print('최종 P/R/F1:', eda_summary['final_metrics_policy'])
print('품질 요약:', eda_summary['data_quality'])
print('LLM 요약:', eda_summary['methods']['llm_only'])
print('Hybrid 요약:', eda_summary['methods']['hybrid'])
print('그래프 수:', len(eda_summary['plots']))

통합 상태: EXECUTED_READ_ONLY
기관별 통합 건수: {'KDI': 160, 'BOK': 160}
전체 corpus: 3539
원문 대조: {'KDI': {'EXACT_MATCH': 160}, 'BOK': {'EXACT_MATCH': 160}}
KDI A/B: 39 일치, 10 불일치, 일치율=79.59%
BOK A/B: 68 일치, 92 불일치, 일치율=42.50%
ALL A/B: 107 일치, 102 불일치, 일치율=51.20%
독립 Gold: {'status': 'NOT_READY', 'count': 0, 'path': 'reports/independent_gold_test.csv'}
신규 API 호출: 0
기록된 API 호출: 20
방법별 상태/커버리지: {'keyword_only': ('EXECUTED_OFFLINE', 320), 'llm_only': ('EXECUTED_LLM_PILOT', 20), 'hybrid': ('EXECUTED_LLM_PILOT', 279)}


[{'source': 'KDI',
  'document_id': 'kdi_dic_723',
  'title': '기준금리',
  'label_a': '2',
  'label_b': '1',
  'reason': '설명에 인플레이션 및 물가 핵심 개념이 포함되어 있어 2점(직접 관련)으로 판정함.',
  'keyword_label': '',
  'sampling_bucket': '기초',
  'data_quality_status': 'NORMAL',
  'b_review': 'Y',
  'review_status': 'PENDING_HUMAN_REVIEW'},
 {'source': 'KDI',
  'document_id': 'kdi_dic_2703',
  'title': '파일럿 샵',
  'label_a': '1',
  'label_b': '0',
  'reason': '금리, 통화 등 거시경제 지표로서 인플레이션 이해에 간접적 도움을 주므로 1점으로 판정함.',
  'keyword_label': '',
  'sampling_bucket': '기초',
  'data_quality_status': 'NORMAL',
  'b_review': 'Y',
  'review_status': 'PENDING_HUMAN_REVIEW'},
 {'source': 'KDI',
  'document_id': 'kdi_dic_592',
  'title': '국제피셔효과',
  'label_a': '2',
  'label_b': '1',
  'reason': '설명에 인플레이션 및 물가 핵심 개념이 포함되어 있어 2점(직접 관련)으로 판정함.',
  'keyword_label': '',
  'sampling_bucket': '기초',
  'data_quality_status': 'NORMAL',
  'b_review': 'Y',
  'review_status': 'PENDING_HUMAN_REVIEW'},
 {'source': 'KDI',
  'document_id': 'kdi_dic

[{'source': 'KDI',
  'document_id': 'kdi_dic_2855',
  'title': '행동주의 펀드',
  'source_match_type': 'EXACT_MATCH',
  'content_hash': 'a97edf5c9dde751c5a70025c29653825cde2ef19c01113dbc2bfbc12947dd4b9',
  'keyword_strategy': 'weighted_broad_candidate_filter',
  'keyword_candidate': 'False',
  'keyword_match_score': '0.0',
  'keyword_score_is_not_relevance': 'True',
  'matched_keywords': '[]',
  'matched_keyword_types': '[]',
  'exclude_keyword_matches': '[]',
  'evidence_span': '',
  'reference_label': '0',
  'reference_label_provenance': 'EXPLICIT_WORKBOOK_LABEL',
  'review_status': 'PENDING'},
 {'source': 'KDI',
  'document_id': 'kdi_dic_723',
  'title': '기준금리',
  'source_match_type': 'EXACT_MATCH',
  'content_hash': '6b29e1ffc9b668f5b8fc1b76d832b2ab2f09efbd521b9570280ad04fda4ba2cd',
  'keyword_strategy': 'weighted_broad_candidate_filter',
  'keyword_candidate': 'False',
  'keyword_match_score': '0.0',
  'keyword_score_is_not_relevance': 'True',
  'matched_keywords': '[]',
  'matched_keyw

최종 P/R/F1: NOT_EVALUATED_UNTIL_INDEPENDENT_GOLD_APPROVED
품질 요약: {'KDI': {'missing_document_id': 0, 'missing_title': 0, 'empty_content': 0, 'duplicate_document_id': 0, 'duplicate_content': 0, 'source_locator_missing': 0, 'quality_distribution': {'정상': 153, '검토필요': 7}, 'original_quality_flag_count': 0, 'content_hash_duplicate_rows': 0, 'reconciliation_counts': {'EXACT_MATCH': 160}, 'explicit_gold_numeric_count': 160, 'explicit_gold_nonnumeric_values': {}}, 'BOK': {'missing_document_id': 0, 'missing_title': 0, 'empty_content': 0, 'duplicate_document_id': 0, 'duplicate_content': 0, 'source_locator_missing': 0, 'quality_distribution': {'정상': 150, '검토필요': 10}, 'original_quality_flag_count': 10, 'content_hash_duplicate_rows': 0, 'reconciliation_counts': {'EXACT_MATCH': 160}, 'explicit_gold_numeric_count': 0, 'explicit_gold_nonnumeric_values': {'완료': 160}}}
LLM 요약: {'status': 'EXECUTED_LLM_PILOT', 'evaluated_docs': 20, 'coverage_rate': 0.0625, 'proxy_question': {'status': 'DRY_RUN_AI_REFERENCE

## 8. LLM-only 결과

저장된 파일럿 결과만 사용하며, 실제 처리 건수·모델·prompt version·JSON/근거 검증 실패·지연시간·토큰·비용 상태는 `precision_eda_summary.json`에서 읽는다. 현재 파일럿은 최대 20건이며 320건 전체 실행으로 해석하지 않는다.

## 9. Hybrid 결과

Keyword 후보와 LLM 파일럿 결과를 결합하되 Keyword 미매칭 문서는 평가 모집단에서 제거하지 않는다. LLM 결과가 없는 후보는 보류 상태로 유지한다.

## 10. Precision / Recall / F1

문제 생성은 관련도 2, Tutor 선별은 관련도 1·2를 양성으로 사용한다. 독립 Gold 승인 전에는 proxy 일치도만 표시하고 최종 TP/FP/FN/TN 및 Precision·Recall·F1은 `NOT_EVALUATED`로 유지한다.

## 11. 세 가지 방식 비교

평가 문서 수와 전체 320건 대비 커버리지를 함께 비교한다. LLM-only와 Hybrid의 관측 모수가 Keyword-only와 다르므로 Gold 확보 전에는 성능 순위를 주장하지 않는다.

## 12. 시각화

실행 시 `reports/evaluation/`에 데이터 규모, 라벨 분포, A/B 교차표·Confusion Matrix, 방법별 분포·일치율, API 지연·토큰 그래프를 생성한다. 독립 Gold 그래프는 Gold 확보 전 생성하지 않는다.

## 13. 주요 사례 분석

키워드 후보/reference 불일치, A/B 불일치, LLM·Hybrid 관측 차이는 CSV와 JSON 사례 목록으로 확인한다. 독립 Gold 전에는 오분류가 아닌 `판정 불일치 사례`로 표시한다.

## 14. 연구 결과 요약

최신 실행 수치는 `reports/evaluation/precision_eda_summary.json`과 `precision_eda_report.md`에 저장된다. API 호출은 첫 셀의 `RUN_LLM_API=True`일 때만 최대 20건 실행되며, 독립 Gold 미확보 상태에서는 호출 결과도 예비 분석으로만 사용한다.